# Валидация и кросс-валидация — простыми словами

Это **учебный ноутбук**: зачем делить данные, почему «подглядывать в Test» вредно, как работает **кросс-валидация** и где прячется **утечка данных**.

Здесь **нет** страшной математики. Идём маленькими шагами: аналогия → картинка в таблице → код в `sklearn` → шпаргалка.

**Что понадобится**
- Python + Jupyter / VS Code / Colab
- Библиотеки: `numpy`, `pandas`, `matplotlib`, `scikit-learn`

Запускайте ячейки **по порядку** (сверху вниз).

---

## План

1. [Словарь терминов](#dict)
2. [Train / Validation / Test — учебник, пробник, экзамен](#splits)
3. [Зачем нужна валидационная выборка](#why-val)
4. [Что такое кросс-валидация](#cv)
5. [K-Fold на пальцах (маленький пример)](#kfold-hand)
6. [K-Fold в sklearn](#kfold-code)
7. [Почему после CV обучают модель заново](#retrain)
8. [Для чего используют CV](#uses)
9. [Опасность: утечка данных (data leakage)](#leakage)
10. [Правильный способ: Pipeline](#pipeline)
11. [Два способа передать `cv` в `cross_val_score`](#cv-api)
12. [Основные виды кросс-валидации](#types)
13. [Плюсы и минусы](#proscons)
14. [Самое главное + шпаргалка](#итог)
15. [Мини-практика](#practice)


<a id="dict"></a>
## 1. Словарь (обязательно прочитать)

| Термин | Простыми словами |
|--------|------------------|
| **Датасет** | Таблица с данными |
| **Объект / sample** | Одна строка (одна квартира, один клиент, один день) |
| **Признак / feature** | Столбец-вход: площадь, возраст, температура… |
| **Target / y** | То, что предсказываем (цена, класс, продажи) |
| **Модель** | Алгоритм, который учится по примерам и предсказывает |
| **Обучение (`fit`)** | «Посмотри на данные и подбери параметры» |
| **Предсказание (`predict`)** | «Дай ответ для новых объектов» |
| **Метрика** | Число «насколько модель ошибается / насколько хороша» (MSE, MAE, accuracy…) |
| **Train** | Данные, на которых модель **учится** |
| **Validation (valid)** | Данные для **выбора** модели и настроек (пробник) |
| **Test** | Данные для **финальной** честной проверки (экзамен) |
| **Гиперпараметр** | Настройка «снаружи» модели: глубина дерева, `K` в KNN, степень полинома… |
| **Кросс-валидация (CV)** | Несколько раз учим и проверяем на **разных** частях Train |
| **Фолд (fold)** | Одна «доля» данных в K-Fold |
| **Утечка данных (data leakage)** | В проверку случайно «подсмотрели» информацию, которой в реальной жизни не будет |
| **Pipeline** | Конвейер шагов: предобработка + модель, чтобы всё делалось **честно внутри** каждого фолда |
| **Shuffle** | Перемешать строки перед разбиением |


<a id="splits"></a>
## 2. Train / Validation / Test — учебник, пробник, экзамен

Самая важная аналогия всего урока:

| Часть данных | Аналогия | Зачем |
|--------------|----------|--------|
| **Train** | **Учебник** | Учимся: читаем задачи, запоминаем закономерности |
| **Validation** | **Пробник** / контрольная для тренировки | Пробуем разные стратегии, смотрим, что работает лучше |
| **Test** | **Экзамен** | Один раз проверяем итог. На экзамене уже **нельзя** «подкручивать» подготовку |

### Простыми словами

1. На **Train** модель **учится**.
2. На **Validation** мы **выбираем**: какая модель / какие настройки лучше.
3. На **Test** мы **один раз** отвечаем: «а насколько хорошо получилось *на самом деле*?»

> **Test должен оставаться неизвестным** до финальной проверки.  
> Если мы снова и снова смотрим на Test и под него подстраиваемся — это уже не экзамен, а «репетиция с ответами».

### Типичные пропорции (ориентир, не закон)

| Схема | Пример |
|-------|--------|
| Train + Test | 80% / 20% |
| Train + Valid + Test | 60% / 20% / 20% или 70% / 15% / 15% |
| Train с CV + Test | 80% (внутри — кросс-валидация) / 20% Test |

Точные проценты зависят от размера данных. На очень маленьких таблицах чаще помогает **кросс-валидация** (раздел 4).


<a id="why-val"></a>
## 3. Зачем нужна валидационная выборка?

### Плохой сценарий (очень частый)

Каждый раз мы смотрим **только на Test** и говорим:

> «Эта модель лучше.»

Потом чуть меняем модель → снова смотрим Test → снова меняем → снова смотрим Test…

**Что случилось?**  
Test начал **влиять на процесс разработки**.  
Он перестал быть «полностью неизвестным экзаменом».

Модель (и мы сами) постепенно **подстраиваются** под Test — даже если мы не клали эти строки в `fit`.  
Оценка на Test становится **слишком оптимистичной**. В бою / на новых данных результат часто хуже.

### Когда Validation *формально* не нужна?

Если вы **один раз** обучили 2–3 модели, **один раз** сравнили и **больше ничего не крутите** — теоретически можно сравнить на Test.

Но в реальной жизни почти всегда цикл такой:

```text
посмотрел на результат → изменил модель → снова посмотрел → снова изменил ...
```

Именно для этого цикла нужна **Validation** (или кросс-валидация внутри Train).

### Схема «правильно»

```text
[ Все данные ]
      |
      +--> Test (отложили и НЕ трогаем до финала)
      |
      +--> Train
              |
              +--> учим кандидатов
              +--> сравниваем на Validation (или через CV)
              +--> выбираем лучшего
      |
      v
Финальная модель → один раз оцениваем на Test
```

> **Train** — учебник. **Validation** — пробник. **Test** — экзамен.


### Мини-демо: «подглядывание в Test» завышает оценку

Сгенерируем данные, «подберём» степень полинома, глядя **на test**, и сравним с честным выбором **на validation**.

Идея эксперимента:
1. Есть train / valid / test.
2. **Нечестный** путь: выбираем степень полинома по ошибке на **test**.
3. **Честный** путь: выбираем степень по ошибке на **valid**, а test смотрим **один раз** в конце.


In [13]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(42)

# --- «настоящая» зависимость: кривая + шум ---
n = 80
X = np.linspace(0, 1, n).reshape(-1, 1)
y = np.sin(2 * np.pi * X.ravel()) + np.random.normal(0, 0.25, size=n)

# Сначала отложим Test, потом от Train отрежем Valid
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size=0.33, random_state=1)

print(f"Train: {len(X_train)}, Valid: {len(X_valid)}, Test: {len(X_test)}")

degrees = [1, 2, 3, 5, 8, 12]
rows = []

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())
    model.fit(X_train, y_train)
    mse_tr = mean_squared_error(y_train, model.predict(X_train))
    mse_va = mean_squared_error(y_valid, model.predict(X_valid))
    mse_te = mean_squared_error(y_test, model.predict(X_test))
    rows.append((d, mse_tr, mse_va, mse_te))

print(f"{'degree':>6} | {'MSE train':>10} | {'MSE valid':>10} | {'MSE test':>10}")
print("-" * 48)
for d, tr, va, te in rows:
    print(f"{d:6d} | {tr:10.4f} | {va:10.4f} | {te:10.4f}")

# Нечестный выбор: лучший по TEST
best_cheat = min(rows, key=lambda r: r[3])
# Честный выбор: лучший по VALID
best_honest = min(rows, key=lambda r: r[2])

print()
print(f"Нечестно (минимум на TEST):  degree={best_cheat[0]}, "
      f"MSE test={best_cheat[3]:.4f}  ← мы уже «подсмотрели» test при выборе")
print(f"Честно (минимум на VALID):   degree={best_honest[0]}, "
      f"MSE valid={best_honest[2]:.4f}, MSE test={best_honest[3]:.4f}")
print()
print("Смысл: test в честном сценарии НЕ участвовал в выборе degree.")
print("В нечестном — участвовал. На реальных задачах это часто даёт завышенный оптимизм.")


Train: 40, Valid: 20, Test: 20
degree |  MSE train |  MSE valid |   MSE test
------------------------------------------------
     1 |     0.1760 |     0.3197 |     0.2118
     2 |     0.1722 |     0.3197 |     0.2128
     3 |     0.0419 |     0.1302 |     0.0608
     5 |     0.0380 |     0.1053 |     0.0520
     8 |     0.0348 |     0.1106 |     0.0446
    12 |     0.0323 |     0.1300 |     0.0525

Нечестно (минимум на TEST):  degree=8, MSE test=0.0446  ← мы уже «подсмотрели» test при выборе
Честно (минимум на VALID):   degree=5, MSE valid=0.1053, MSE test=0.0520

Смысл: test в честном сценарии НЕ участвовал в выборе degree.
В нечестном — участвовал. На реальных задачах это часто даёт завышенный оптимизм.


**Что заметить в таблице**

- На **train** очень гибкая модель (большая степень) часто даёт **маленькую** ошибку — она может «вызубрить» шум.
- **Valid** и **test** обычно растут, когда модель слишком сложная (переобучение).
- Если выбирать степень **по test**, мы используем экзамен как пробник — оценка на test перестаёт быть независимой.

Дальше — как сделать пробник **надёжнее**, чем одно случайное разбиение 80/20.


<a id="cv"></a>
## 4. Что такое кросс-валидация?

**Кросс-валидация (Cross-Validation, CV)** — способ **несколько раз** обучить и проверить модель на **разных частях** обучающих данных, чтобы получить **более надёжную** оценку качества.

### Главная идея

Мы **не доверяем** результату **одной** случайной валидационной выборки.  
Поэтому несколько раз **меняем**, какие объекты в обучении, а какие — в проверке.

### Почему одного Validation мало?

Допустим, разделили один раз:

- Train — 80%
- Validation — 20%

Может **случайно** получиться, что в Validation попали:

1. слишком **сложные** объекты;
2. слишком **простые** объекты;
3. много **выбросов**;
4. объекты, **плохо представляющие** весь датасет.

Тогда оценка зависит не только от качества модели, но и от **удачи разбиения**.

**Кросс-валидация уменьшает влияние этой случайности.**

### Важный принцип

> Кросс-валидацию обычно проводят **только внутри Train**.  
> **Test в CV не участвует.**

Иначе снова «подглядываем в экзамен».


<a id="kfold-hand"></a>
## 5. K-Fold на пальцах

Самый распространённый вариант — **K-Fold**.

### Как устроено

1. Все **тренировочные** данные делят на **K** примерно равных частей (**фолдов**).
2. Делают **K** обучений.
3. На каждом шаге **один** фолд — Validation, остальные **K−1** — Train.
4. Считают метрику на каждом валидационном фолде.
5. Итог: **среднее** по K значениям (и часто ещё **разброс**).

### Пример при K = 5

| Раунд | Validation | Train |
|-------|------------|-------|
| 1 | Fold 1 | Fold 2, 3, 4, 5 |
| 2 | Fold 2 | Fold 1, 3, 4, 5 |
| 3 | Fold 3 | Fold 1, 2, 4, 5 |
| 4 | Fold 4 | Fold 1, 2, 3, 5 |
| 5 | Fold 5 | Fold 1, 2, 3, 4 |

В результате:

- каждый объект **один раз** побывает в Validation;
- и **K−1** раз — в Train.

Если метрика — MSE, получим 5 чисел MSE → берём **среднее**.

> Именно **средний** MSE (или другая метрика) по фолдам обычно используют для **сравнения моделей**.  
> Полезно смотреть и **разброс**: среднее = «типичное качество», разброс = «насколько стабильно».

### Картинка «в одну строку»

```text
Данные Train:  [ 1 | 2 | 3 | 4 | 5 ]   ← 5 фолдов

Раунд 1:       [ V | T | T | T | T ]
Раунд 2:       [ T | V | T | T | T ]
Раунд 3:       [ T | T | V | T | T ]
Раунд 4:       [ T | T | T | V | T ]
Раунд 5:       [ T | T | T | T | V ]

Итог = mean(score1 … score5)
```


### Ручной мини-пример: 6 точек и K = 3

Возьмём крошечные данные, чтобы **видеть глазами**, кто в каком фолде.

Задача-игрушка: предсказать `y` по `x` (линейная регрессия).  
Метрика — **MAE** (средняя абсолютная ошибка), её проще читать, чем MSE.


In [14]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 6 «квартир»: площадь → цена (упрощённо)
df = pd.DataFrame({
    "id":   [1, 2, 3, 4, 5, 6],
    "area": [30, 40, 50, 60, 70, 80],
    "price": [3.2, 4.1, 5.0, 6.3, 6.8, 8.1],  # млн ₽, выдумано
})
print("Все данные (это наш Train — Test отдельно не трогаем):")
print(df.to_string(index=False))

# K = 3 → в каждом фолде по 2 объекта (по порядку id, без shuffle — для наглядности)
folds = [
    df.iloc[0:2],  # fold 0
    df.iloc[2:4],  # fold 1
    df.iloc[4:6],  # fold 2
]

print("\nФолды:")
for i, f in enumerate(folds):
    print(f"  Fold {i}: id = {list(f['id'])}")

scores = []
print("\n--- Раунды K-Fold ---")
for i in range(3):
    valid = folds[i]
    train = pd.concat([folds[j] for j in range(3) if j != i], ignore_index=True)

    X_tr = train[["area"]].values
    y_tr = train["price"].values
    X_va = valid[["area"]].values
    y_va = valid["price"].values

    model = LinearRegression()
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    mae = mean_absolute_error(y_va, pred)
    scores.append(mae)

    print(f"\nРаунд {i+1}: valid id={list(valid['id'])}, train id={list(train['id'])}")
    print(f"  a≈{model.coef_[0]:.3f}, b≈{model.intercept_:.3f}")
    print(f"  правда:      {np.round(y_va, 2)}")
    print(f"  предсказание:{np.round(pred, 2)}")
    print(f"  MAE на valid: {mae:.3f}")

print("\n" + "=" * 40)
print(f"MAE по фолдам: {np.round(scores, 3)}")
print(f"Средний MAE:   {np.mean(scores):.3f}  ← оценка качества через CV")
print(f"Разброс (std): {np.std(scores):.3f}  ← насколько нестабильно между фолдами")


Все данные (это наш Train — Test отдельно не трогаем):
 id  area  price
  1    30    3.2
  2    40    4.1
  3    50    5.0
  4    60    6.3
  5    70    6.8
  6    80    8.1

Фолды:
  Fold 0: id = [1, 2]
  Fold 1: id = [3, 4]
  Fold 2: id = [5, 6]

--- Раунды K-Fold ---

Раунд 1: valid id=[1, 2], train id=[3, 4, 5, 6]
  a≈0.098, b≈0.180
  правда:      [3.2 4.1]
  предсказание:[3.12 4.1 ]
  MAE на valid: 0.040

Раунд 2: valid id=[3, 4], train id=[1, 2, 5, 6]
  a≈0.096, b≈0.276
  правда:      [5.  6.3]
  предсказание:[5.07 6.03]
  MAE на valid: 0.171

Раунд 3: valid id=[5, 6], train id=[1, 2, 3, 4]
  a≈0.102, b≈0.060
  правда:      [6.8 8.1]
  предсказание:[7.2  8.22]
  MAE на valid: 0.260

MAE по фолдам: [0.04  0.171 0.26 ]
Средний MAE:   0.157  ← оценка качества через CV
Разброс (std): 0.090  ← насколько нестабильно между фолдами


**Мини-вывод**

1. На каждом раунде — **своя** временная модель (свои `a` и `b`).
2. Оценка CV — это **среднее** ошибок на разных valid-фолдах, а не «один удачный сплит».
3. `std` по фолдам показывает: модель **стабильна** или «прыгает» от куска данных к куску.

Ниже — то же самое «по-взрослому» через `sklearn`.


<a id="kfold-code"></a>
## 6. K-Fold в sklearn

Два удобных способа:

1. **`KFold`** — явно получить индексы фолдов (удобно для понимания и отладки).
2. **`cross_val_score`** — сразу получить оценки по фолдам (удобно в работе).

### Про `scoring="neg_mean_squared_error"`

В `sklearn` для `cross_val_score` принято: **чем больше score — тем лучше**.  
MSE — «чем меньше, тем лучше», поэтому берут **отрицательный** MSE: `neg_mean_squared_error`.

Чтобы получить обычный MSE:

```python
mean_mse = -scores.mean()
```


### Подсказка наперёд

Параметр `cv` можно передать **двумя способами**:

1. объектом `KFold(...)` / `StratifiedKFold(...)` — полный контроль (`shuffle`, `random_state`);
2. числом `cv=5` — коротко, но **по умолчанию без перемешивания**.

Подробно — в [§11](#cv-api) после Pipeline.


In [15]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error

np.random.seed(42)

# Синтетическая регрессия: 100 объектов, 3 признака
X, y = make_regression(n_samples=100, n_features=3, noise=15.0, random_state=42)

# 1) Сначала навсегда откладываем Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} объектов, Test: {X_test.shape[0]} объектов")
print("Test дальше в CV НЕ используем.\n")

# 2) K-Fold только внутри Train
cv = KFold(n_splits=5, shuffle=True, random_state=42)
# shuffle=True — перемешать строки перед нарезкой на фолды
# random_state=42 — чтобы у всех был одинаковый результат

model = LinearRegression()

# Способ А: cross_val_score
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_squared_error",
)

mse_folds = -scores  # вернули знак
print("MSE по 5 фолдам:", np.round(mse_folds, 2))
print(f"Средний MSE (CV): {mse_folds.mean():.2f}")
print(f"Std MSE (CV):     {mse_folds.std():.2f}")

# Способ Б: руками через split — видно индексы
print("\nИндексы valid-фолдов (первые 5 id каждого):")
for fold_i, (tr_idx, va_idx) in enumerate(cv.split(X_train), start=1):
    print(f"  Fold {fold_i}: n_train={len(tr_idx)}, n_valid={len(va_idx)}, valid_idx[:5]={va_idx[:5]}")


Train: 80 объектов, Test: 20 объектов
Test дальше в CV НЕ используем.

MSE по 5 фолдам: [182.56 209.19 240.08 247.46 269.25]
Средний MSE (CV): 229.71
Std MSE (CV):     30.43

Индексы valid-фолдов (первые 5 id каждого):
  Fold 1: n_train=64, n_valid=16, valid_idx[:5]=[ 0  4 10 12 18]
  Fold 2: n_train=64, n_valid=16, valid_idx[:5]=[ 5  9 16 34 39]
  Fold 3: n_train=64, n_valid=16, valid_idx[:5]=[ 3  6  7  8 13]
  Fold 4: n_train=64, n_valid=16, valid_idx[:5]=[11 15 24 26 27]
  Fold 5: n_train=64, n_valid=16, valid_idx[:5]=[ 1  2 14 20 21]


<a id="retrain"></a>
## 7. Почему после кросс-валидации модель обучают заново?

Во время CV создаётся **не одна** итоговая модель, а **несколько временных**.

При `K = 5`:

- модель обучается **5 раз**;
- каждый раз — примерно на **80%** Train (если K=5);
- эти модели нужны для **оценки**, а не для продакшена.

### Что делать после выбора настроек?

1. Выбрали тип модели / гиперпараметры / предобработку **по среднему CV-score**.
2. Создаём **новую** итоговую модель с этими настройками.
3. Обучаем её на **100% Train**.
4. **Один раз** проверяем на **Test**.

```text
CV:     5 временных моделей на 4/5 Train  → только для оценки
Итог:   1 модель на всём Train            → для использования
Экзамен: 1 раз predict на Test
```

> Аналогия: на пробниках вы тренируетесь разными способами.  
> На экзамен идёте **одним** лучшим способом, выучив **весь** учебник, а не 80%.


In [16]:
# Продолжаем данные из предыдущей ячейки (X_train, y_train, X_test, y_test)

# 1) Оценка через CV (временные модели внутри cross_val_score)
cv_scores = cross_val_score(
    LinearRegression(),
    X_train,
    y_train,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
)
print(f"CV mean MSE: {-cv_scores.mean():.2f}")

# 2) Итоговая модель — заново на 100% Train
final_model = LinearRegression()
final_model.fit(X_train, y_train)

# 3) Честный экзамен — один раз на Test
test_mse = mean_squared_error(y_test, final_model.predict(X_test))
print(f"Test MSE (один раз): {test_mse:.2f}")
print()
print("CV ≈ ожидание качества при выборе настроек.")
print("Test = финальный отчёт. Его не используют, чтобы снова крутить модель.")


CV mean MSE: 229.71
Test MSE (один раз): 278.66

CV ≈ ожидание качества при выборе настроек.
Test = финальный отчёт. Его не используют, чтобы снова крутить модель.


<a id="uses"></a>
## 8. Для чего используют кросс-валидацию?

CV помогает **выбрать** (по среднему качеству на фолдах):

1. **тип модели** (линейная / дерево / …);
2. **глубину дерева**;
3. **количество деревьев**;
4. **коэффициент регуляризации**;
5. **число соседей** в KNN;
6. **способ заполнения пропусков** (mean / median / …);
7. **набор признаков**;
8. **степень полинома**;
9. другие **гиперпараметры** и шаги **предобработки**.

### Правило сравнения

> Сравниваем кандидатов по **среднему** score по фолдам,  
> а **не** по одному «самому удачному» фолду.

Иначе снова ловим случайность.


### Мини-пример: выбираем степень полинома через CV (честно)

Test снова **не участвует** в выборе — только в финале.


In [17]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

np.random.seed(7)
X_poly = np.linspace(-1, 1, 60).reshape(-1, 1)
y_poly = 0.5 * X_poly.ravel()**3 - 0.3 * X_poly.ravel() + np.random.normal(0, 0.08, size=60)

X_tr, X_te, y_tr, y_te = train_test_split(X_poly, y_poly, test_size=0.25, random_state=0)

cv = KFold(n_splits=5, shuffle=True, random_state=0)
degrees = [1, 2, 3, 5, 8]
cv_means = []

print(f"{'degree':>6} | {'CV mean MSE':>12} | {'CV std':>8}")
print("-" * 36)
for d in degrees:
    pipe = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())
    sc = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="neg_mean_squared_error")
    mse = -sc
    cv_means.append(mse.mean())
    print(f"{d:6d} | {mse.mean():12.5f} | {mse.std():8.5f}")

best_d = degrees[int(np.argmin(cv_means))]
print(f"\nЛучшая degree по CV: {best_d}")

# Итоговая модель на всём Train + один test
final = make_pipeline(PolynomialFeatures(best_d, include_bias=False), LinearRegression())
final.fit(X_tr, y_tr)
print(f"Test MSE итоговой модели: {mean_squared_error(y_te, final.predict(X_te)):.5f}")


degree |  CV mean MSE |   CV std
------------------------------------
     1 |      0.01337 |  0.00368
     2 |      0.01402 |  0.00378
     3 |      0.00937 |  0.00153
     5 |      0.01134 |  0.00303
     8 |      0.01516 |  0.00436

Лучшая degree по CV: 3
Test MSE итоговой модели: 0.00464


<a id="leakage"></a>
## 9. Опасность: утечка данных (data leakage)

### Что это такое?

**Утечка данных** — когда в обучение или в подготовку признаков попадает информация,  
которой **не должно быть** на момент предсказания (в том числе из valid/test).

Модель «видит будущее» или «подсматривает ответ» → метрики красивые, в бою — плохо.

### Классическая ошибка с пропусками

Представим: **перед** кросс-валидацией заполнили пропуски средним, посчитанным по **всему** `X_train`:

```python
X_train = imputer.fit_transform(X_train)   # среднее по ВСЕМУ train
cross_val_score(model, X_train, y_train, cv=5)
```

**Почему это нечестно?**  
Во время проверки одного фолда среднее уже посчитано **с учётом объектов этого фолда**.  
Валидационная часть **чуть-чуть повлияла** на обработку «тренировочной» части.

Это и есть **data leakage** (здесь — утечка через статистику предобработки).

### Когда утечки от обычного K-Fold **нет**?

Если одновременно:

1. **нет** предобработки, которая считает статистики по данным;
2. **нет** вычисления mean/std/min/max/target encoding «на всём train сразу»;
3. вы **просто** передаёте `X_train` в модель,

то сам по себе `KFold` утечку **не создаёт**: в каждом раунде `fit` видит только train-часть фолда.

### Когда утечка появляется?

Когда **перед кросс-валидацией** ты делаешь что-то, что «смотрит» на **все тренировочные данные сразу**.

Типичные случаи:

- `StandardScaler.fit` на всём train, потом CV;
- `SimpleImputer.fit` на всём train, потом CV;
- target encoding по всему train, потом CV;
- отбор признаков с использованием **всего** train (и тем более y) **снаружи** CV;
- любое использование **Test** для выбора модели.

> Коротко: всё, что **fit**’ится на данных, должно fit’иться **только на train-части текущего фолда**.


### Демонстрация: «заранее заполнили среднее» vs честный Pipeline

Сделаем данные **с пропусками** и сравним два подхода.  
На маленькой синтетике разница может быть скромной, но **принцип** тот же, что в больших задачах.


In [18]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

np.random.seed(0)
n = 120
rng = np.random.default_rng(0)

X_full = rng.normal(size=(n, 2))
# y зависит от признаков
y = 3 * X_full[:, 0] - 2 * X_full[:, 1] + rng.normal(0, 0.5, size=n)

# Искусственно делаем пропуски в 25% ячеек первого признака
X_miss = X_full.copy()
mask = rng.random(n) < 0.25
X_miss[mask, 0] = np.nan

print(f"Пропусков в признаке 0: {np.isnan(X_miss[:, 0]).sum()} из {n}")

cv = KFold(n_splits=5, shuffle=True, random_state=0)

# --- НЕПРАВИЛЬНО: imputer на всём X заранее ---
X_bad = SimpleImputer(strategy="mean").fit_transform(X_miss)  # видел ВСЕ строки
scores_bad = cross_val_score(
    LinearRegression(), X_bad, y, cv=cv, scoring="neg_mean_squared_error"
)

# --- ПРАВИЛЬНО: imputer внутри Pipeline (fit только на train-фолде) ---
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", LinearRegression()),
])
scores_good = cross_val_score(
    pipe, X_miss, y, cv=cv, scoring="neg_mean_squared_error"
)

print(f"\nНеправильный путь (impute на всём X, потом CV): mean MSE = {-scores_bad.mean():.4f}")
print(f"Правильный путь (Pipeline внутри CV):           mean MSE = {-scores_good.mean():.4f}")
print()
print("Даже если числа близки, неправильный путь логически «подмешивает» valid в статистики.")
print("На target encoding / отборе признаков / масштабировании ошибка бывает гораздо заметнее.")


Пропусков в признаке 0: 27 из 120

Неправильный путь (impute на всём X, потом CV): mean MSE = 1.4873
Правильный путь (Pipeline внутри CV):           mean MSE = 1.4769

Даже если числа близки, неправильный путь логически «подмешивает» valid в статистики.
На target encoding / отборе признаков / масштабировании ошибка бывает гораздо заметнее.


<a id="pipeline"></a>
## 10. Правильный способ: `Pipeline`

`Pipeline` склеивает шаги: **сначала** предобработка, **потом** модель.

Внутри `cross_val_score` для **каждого** фолда происходит отдельно:

1. вычислить статистики (mean и т.д.) **только на train-части фолда**;
2. применить предобработку к train и valid;
3. обучить модель на train-части;
4. оценить на valid-части.

Это **честный** вариант.


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

# Тот же X_miss, y из предыдущей ячейки

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),  # заполнить пропуски
    ("scaler", StandardScaler()),                 # нормировка (тоже только по train-фолду!)
    ("model", LinearRegression()),
])

scores = cross_val_score(
    pipeline,
    X_miss,
    y,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
)

mean_mse = -scores.mean()
print("MSE по фолдам:", np.round(-scores, 4))
print(f"Средний MSE: {mean_mse:.4f}")
print()
print("Цепочка в каждом фолде: fit imputer → fit scaler → fit model на train-части;")
print("transform + predict на valid-части. Valid не участвует в fit предобработки.")


MSE по фолдам: [1.8583 1.3871 1.1895 0.5624 2.3072]
Средний MSE: 1.4609

Цепочка в каждом фолде: fit imputer → fit scaler → fit model на train-части;
transform + predict на valid-части. Valid не участвует в fit предобработки.


### Схема утечки vs Pipeline

```text
НЕПРАВИЛЬНО
[ весь X_train ] --fit imputer--> [ заполненный X_train ] --CV--> оценки
                     ↑
         valid-фолд уже «засветился» в mean

ПРАВИЛЬНО (Pipeline + CV)
для каждого фолда:
   train_fold --fit imputer/scaler/model--> 
   valid_fold --transform + predict--> score
```


<a id="cv-api"></a>
## 11. Два способа передать `cv` в `cross_val_score`

После того как `Pipeline` готов, оценки через CV можно получить **двумя** стилями записи.  
Смысл один — **K фолдов**, но **контроль над разбиением** разный.

---

### Вариант 1. Сами создаём `KFold` (полный контроль)

```python
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=kf,                      # ← объект разбиения
    scoring="neg_mean_squared_error"
)
```

Здесь вы **сами** создаёте объект `KFold` и **полностью управляете**, как режутся данные.

Можно явно указать:

| Параметр | Смысл |
|----------|--------|
| `n_splits=5` | число фолдов (K) |
| `shuffle=True` | **перемешать** строки перед разбиением |
| `random_state=42` | чтобы разбиение **всегда было одинаковым** (воспроизводимость) |

**Когда так лучше:** почти всегда в учебных и рабочих ноутбуках — видно, *как* режем, и результат повторяем.

---

### Вариант 2. Коротко: `cv=5`

```python
scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,                       # ← просто число
    scoring="neg_mean_squared_error"
)
```

`cross_val_score` **сам внутри** создаёт разбиение.

Фактически это **почти** как:

```python
KFold(n_splits=5)
```

#### Важный нюанс

По умолчанию у такого `KFold`:

```python
shuffle=False
```

То есть данные **не** перемешиваются.

| | `cv=5` | `KFold(5, shuffle=True, random_state=42)` |
|--|--------|---------------------------------------------|
| Число фолдов | 5 | 5 |
| Перемешивание | **нет** (по умолчанию) | **да** |
| Воспроизводимый shuffle | — | через `random_state` |

Если строки в таблице идут **блоками** (сначала один тип объектов, потом другой), без `shuffle` фолды могут получиться **неудачными**.  
Для «спокойных» перемешанных данных разница часто небольшая, но **привычка** писать явный `KFold(..., shuffle=True, random_state=...)` безопаснее.

---

### Для классификации (самый частый рабочий шаблон)

Обычный `KFold` **не** следит за долей классов.  
При дисбалансе в одном фолде может почти не оказаться редкого класса.

```python
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=skf,                 # ← stratified!
    scoring="accuracy"      # или f1, roc_auc, ...
)
```

`StratifiedKFold` старается, чтобы в **каждом** фолде было **примерно одинаковое соотношение классов**.

> Регрессия → чаще `KFold`.  
> Классификация → чаще `StratifiedKFold`.  
> В обоих случаях: предобработка **внутри** `Pipeline`, а не до CV.

Ниже — живой код: вариант 1, вариант 2 и stratified для классификации.


In [ ]:
import numpy as np
from sklearn.datasets import make_regression, make_classification
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score, train_test_split
)

# ---------- Регрессия: сравниваем cv=KFold(...) и cv=5 ----------
np.random.seed(0)
X_reg, y_reg = make_regression(n_samples=120, n_features=4, noise=12.0, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=0)

# Нарочно «испорченный» порядок строк: сначала малые y, потом большие
# (без shuffle фолды могут отличаться сильнее)
order = np.argsort(y_tr)
X_tr_ord, y_tr_ord = X_tr[order], y_tr[order]

pipe_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

# Вариант 1: полный контроль
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_v1 = cross_val_score(
    pipe_reg, X_tr_ord, y_tr_ord, cv=kf, scoring="neg_mean_squared_error"
)

# Вариант 2: cv=5 → внутри KFold(n_splits=5), shuffle=False
scores_v2 = cross_val_score(
    pipe_reg, X_tr_ord, y_tr_ord, cv=5, scoring="neg_mean_squared_error"
)

print("Регрессия (данные специально отсортированы по y):")
print(f"  Вариант 1 KFold(shuffle=True):  mean MSE = {-scores_v1.mean():.2f}, std = {(-scores_v1).std():.2f}")
print(f"  Вариант 2 cv=5 (shuffle=False): mean MSE = {-scores_v2.mean():.2f}, std = {(-scores_v2).std():.2f}")
print("  → числа могут различаться: разбиение разное из-за shuffle!")
print()

# ---------- Классификация: StratifiedKFold + Pipeline ----------
X_cls, y_cls = make_classification(
    n_samples=300, n_features=8, n_informative=4,
    weights=[0.85, 0.15], random_state=1
)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=1, stratify=y_cls
)

pipe_cls = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_acc = cross_val_score(
    pipe_cls, Xc_tr, yc_tr, cv=skf, scoring="accuracy"
)
scores_f1 = cross_val_score(
    pipe_cls, Xc_tr, yc_tr, cv=skf, scoring="f1"
)

print("Классификация (дисбаланс ~85/15), StratifiedKFold + Pipeline:")
print(f"  accuracy по фолдам: {np.round(scores_acc, 3)}")
print(f"  mean accuracy = {scores_acc.mean():.3f}")
print(f"  mean F1       = {scores_f1.mean():.3f}")
print()
print("Шаблон на память:")
print("  регрессия:      cv = KFold(n_splits=5, shuffle=True, random_state=42)")
print("  классификация:  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)")
print("  всегда:         Pipeline(предобработка + модель) → cross_val_score(...)")


<a id="types"></a>
## 12. Основные виды кросс-валидации

| Вид | Идея | Когда |
|-----|------|--------|
| **K-Fold** | K частей, по очереди valid | Регрессия, «обычные» таблицы |
| **Stratified K-Fold** | В каждом фолде **похожее** соотношение классов | Классификация (особенно дисбаланс) |
| **Time Series Split** | Учимся на прошлом, проверяем на будущем | Временные ряды |
| **Group K-Fold** | Целая **группа** только в train или только в valid | Несколько строк на одного пациента/клиента |
| **Leave-One-Out (LOO)** | Valid = 1 объект, остальное — train | Очень маленькие данные; дорого |

Ниже — коротко про каждый + мини-код.


### 12.1. K-Fold

Обычное деление на K частей. Чаще для **регрессии**.

```python
from sklearn.model_selection import KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)
```

- `shuffle=True` — перемешать строки **перед** разделением (полезно, если данные шли «по порядку»: сначала один класс, потом другой).
- `random_state` — воспроизводимость.

**Типичные K:** 5 или 10. Больше K → оценка стабильнее, но **дороже** по времени.


In [20]:
from sklearn.model_selection import KFold

X_demo = np.arange(10).reshape(-1, 1)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("K-Fold на 10 объектах (индексы):")
for i, (tr, va) in enumerate(cv.split(X_demo), start=1):
    print(f"  Fold {i}: train={tr}, valid={va}")


K-Fold на 10 объектах (индексы):
  Fold 1: train=[0 2 3 4 5 6 7 9], valid=[1 8]
  Fold 2: train=[1 2 3 4 6 7 8 9], valid=[0 5]
  Fold 3: train=[0 1 3 4 5 6 8 9], valid=[2 7]
  Fold 4: train=[0 1 2 3 5 6 7 8], valid=[4 9]
  Fold 5: train=[0 1 2 4 5 7 8 9], valid=[3 6]


### 12.2. Stratified K-Fold

Для **классификации**: сохраняет **долю классов** в каждом фолде.

Пример: во всём датасете 90% класс 0 и 10% класс 1.  
Stratified K-Fold постарается сделать **примерно так же** в каждом фолде.

Обычный K-Fold теоретически может положить **почти все «редкие» единицы** в один фолд — и оценки «прыгают».

Для обычной **регрессии** stratified обычно **не** используют (там нет классов; есть варианты вроде binning target, но это отдельная история).


### Типичный вызов с `cross_val_score` (запомните шаблон)

```python
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy"
)
```

`StratifiedKFold` следит, чтобы в каждом фолде было **примерно одинаковое соотношение классов**.  
Это **самый распространённый** выбор `cv` для бинарной и многоклассовой классификации.


In [21]:
from sklearn.model_selection import StratifiedKFold, KFold

# 20 объектов: 16 нулей и 4 единицы (дисбаланс)
y_cls = np.array([0] * 16 + [1] * 4)
X_cls = np.arange(len(y_cls)).reshape(-1, 1)

print("Обычный K-Fold (доля класса 1 в valid может гулять):")
for i, (tr, va) in enumerate(KFold(n_splits=4, shuffle=True, random_state=0).split(X_cls, y_cls), 1):
    print(f"  Fold {i}: valid y={y_cls[va]},  доля_1={y_cls[va].mean():.2f}")

print("\nStratifiedKFold (доля класса 1 почти одинаковая):")
for i, (tr, va) in enumerate(StratifiedKFold(n_splits=4, shuffle=True, random_state=0).split(X_cls, y_cls), 1):
    print(f"  Fold {i}: valid y={y_cls[va]},  доля_1={y_cls[va].mean():.2f}")


Обычный K-Fold (доля класса 1 в valid может гулять):
  Fold 1: valid y=[0 0 0 1 1],  доля_1=0.40
  Fold 2: valid y=[0 0 0 0 1],  доля_1=0.20
  Fold 3: valid y=[0 0 0 0 1],  доля_1=0.20
  Fold 4: valid y=[0 0 0 0 0],  доля_1=0.00

StratifiedKFold (доля класса 1 почти одинаковая):
  Fold 1: valid y=[0 0 0 0 1],  доля_1=0.20
  Fold 2: valid y=[0 0 0 0 1],  доля_1=0.20
  Fold 3: valid y=[0 0 0 0 1],  доля_1=0.20
  Fold 4: valid y=[0 0 0 0 1],  доля_1=0.20


### 12.3. Time Series Split

Для **временных рядов**.

Обычный K-Fold опасен: модель может **обучиться на будущем** и проверяться на прошлом.  
В жизни **будущее нельзя** использовать, чтобы предсказать прошлое.

`TimeSeriesSplit` растёт «окошком» вперёд: train всегда **раньше** valid.


In [22]:
from sklearn.model_selection import TimeSeriesSplit

X_time = np.arange(12).reshape(-1, 1)  # как будто дни 0..11

tscv = TimeSeriesSplit(n_splits=4)
print("TimeSeriesSplit (train всегда слева = «прошлое»):")
for i, (tr, va) in enumerate(tscv.split(X_time), 1):
    print(f"  Split {i}: train={tr}, valid={va}")


TimeSeriesSplit (train всегда слева = «прошлое»):
  Split 1: train=[0 1 2 3], valid=[4 5]
  Split 2: train=[0 1 2 3 4 5], valid=[6 7]
  Split 3: train=[0 1 2 3 4 5 6 7], valid=[8 9]
  Split 4: train=[0 1 2 3 4 5 6 7 8 9], valid=[10 11]


### 12.4. Group K-Fold

Когда объекты связаны **группами**.

Пример: у каждого **пациента** несколько медицинских записей.  
Нельзя, чтобы записи **одного** пациента были и в Train, и в Validation: модель может частично **запомнить пациента**, а не болезнь.

`GroupKFold` кладёт **всю группу целиком** только в одну из частей.


In [23]:
from sklearn.model_selection import GroupKFold

# 12 записей, 4 пациента (по 3 записи)
X_g = np.arange(12).reshape(-1, 1)
y_g = np.random.randn(12)
groups = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3])  # id пациента

gkf = GroupKFold(n_splits=4)
print("GroupKFold: пациент целиком в train или valid")
for i, (tr, va) in enumerate(gkf.split(X_g, y_g, groups=groups), 1):
    print(f"  Fold {i}: valid groups={sorted(set(groups[va]))}, "
          f"train groups={sorted(set(groups[tr]))}")


GroupKFold: пациент целиком в train или valid
  Fold 1: valid groups=[np.int64(3)], train groups=[np.int64(0), np.int64(1), np.int64(2)]
  Fold 2: valid groups=[np.int64(2)], train groups=[np.int64(0), np.int64(1), np.int64(3)]
  Fold 3: valid groups=[np.int64(1)], train groups=[np.int64(0), np.int64(2), np.int64(3)]
  Fold 4: valid groups=[np.int64(0)], train groups=[np.int64(1), np.int64(2), np.int64(3)]


### 12.5. Leave-One-Out (LOO)

Каждый раз valid = **один** объект, train = все остальные.

Если объектов 1000 → **1000** обучений.  
Очень **дорого**, поэтому редко; иногда на крошечных датасетах.

```python
from sklearn.model_selection import LeaveOneOut
cv = LeaveOneOut()
```


<a id="proscons"></a>
## 13. Плюсы и минусы кросс-валидации

### Плюсы

1. Более **надёжная** оценка качества модели.
2. Результат **меньше зависит** от одного случайного разбиения.
3. Каждый объект участвует и в обучении, и в проверке (в классическом K-Fold).
4. Особенно полезна при **небольшом** количестве данных.
5. Удобна для выбора **моделей и гиперпараметров**.
6. По **разбросу** (std) видно **стабильность** модели.

### Минусы

1. Модель нужно обучать **несколько** раз.
2. При `cv=5` обучение примерно **в 5 раз** дороже одного fit.
3. При `cv=10` — примерно **в 10 раз** дороже.
4. Нужен **правильный** способ разбиения для времени и групп.
5. Предобработку нужно класть в **Pipeline**, иначе возможна **утечка**.

### CV **не** заменяет Test

Кросс-валидация обычно **заменяет или улучшает** обычную Validation:

```text
Train + Validation + Test
        ↓
Train с кросс-валидацией + Test
```

Логика:

1. **Train-фолды** — модель учится;
2. **Validation-фолды** — выбираем модель и настройки;
3. **Test** — **один раз** проверяет итоговую модель;
4. для сравнения кандидатов — **среднее** по фолдам, не лучший одиночный фолд.


<a id="итог"></a>
## 14. Самое главное

1. **Test — экзамен.** Не крутите модель, глядя на test снова и снова.
2. **Validation / CV — пробник.** На нём выбирают модель, гиперпараметры, предобработку.
3. **Train — учебник.** На нём учат; после выбора настроек — финальный `fit` на **всём** Train.
4. **K-Fold** снижает влияние «неудачного» одного сплита 80/20.
5. Смотрите **среднее и разброс** метрики по фолдам.
6. **Утечка** появляется, когда статистики/encoding/отбор признаков fit’ятся на данных, которые потом идут в valid.
7. Лечение утечки в CV — **`Pipeline`** (и аналогичные честные схемы).
8. Выбирайте **тип CV под природу данных**: stratified / time / group.

### Шпаргалка «что делать»

| Ситуация | Что выбрать |
|----------|-------------|
| Регрессия, обычная таблица | `KFold(shuffle=True)` + Test снаружи |
| Классификация, особенно дисбаланс | `StratifiedKFold` |
| Временной ряд | `TimeSeriesSplit` (не мешать будущее в train) |
| Несколько строк на одного человека/магазин | `GroupKFold` |
| Есть imputer / scaler / PCA / … | Всё это **внутрь** `Pipeline` + `cross_val_score` |
| Выбрали гиперпараметры по CV | Новый `fit` на 100% Train → один раз Test |
| Очень мало данных | CV особенно полезна; LOO — только если данных крохи |
| Нет никакой предобработки | Обычный K-Fold сам по себе не «течёт» |
| Хотите контроль + воспроизводимость | `cv=KFold(..., shuffle=True, random_state=…)` |
| Коротко написали `cv=5` | Внутри `shuffle=False` — помните нюанс |
| Классификация | `cv=StratifiedKFold(..., shuffle=True, random_state=…)` |

### Мини-глоссарий

| Термин | Смысл |
|--------|--------|
| **Train / Valid / Test** | Учебник / пробник / экзамен |
| **CV** | Несколько пробников на разных кусках Train |
| **Fold** | Одна доля в K-Fold |
| **Hyperparameter** | Настройка, которую подбираем снаружи `fit` |
| **Leakage** | Подглядели то, чего не будет в бою |
| **Pipeline** | Предобработка + модель одним честным конвейером |
| **neg_mean_squared_error** | MSE со знаком минус (удобно для sklearn score) |


<a id="practice"></a>
## 15. Мини-практика (попробуйте сами)

### Задание A (понимание)

Ответьте себе (можно в комментариях в следующей ячейке):

1. Чем Validation отличается от Test?
2. Почему CV не заменяет Test?
3. Почему `imputer.fit` на всём Train **до** `cross_val_score` — риск утечки?
4. Зачем после CV делать `fit` на всём Train ещё раз?

### Задание B (код)

На данных ниже:

1. Отложите **Test** (20%).
2. Сравните на **Train** через CV (`cv=5`) две модели:
   - `LinearRegression`
   - `Ridge(alpha=10)`
3. Выберите лучшую по **среднему MSE**.
4. Обучите победителя на **всём Train**.
5. Один раз измерьте MSE на **Test**.

Подсказка: `from sklearn.linear_model import Ridge`


In [24]:
# ===== Мини-практика: шаблон =====
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error

np.random.seed(1)
X, y = make_regression(n_samples=150, n_features=5, noise=20.0, random_state=1)

# TODO-1: train/test split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

cv = KFold(n_splits=5, shuffle=True, random_state=1)

# TODO-2: CV для LinearRegression и Ridge(alpha=10)
models = {
    "LinearRegression": LinearRegression(),
    "Ridge(alpha=10)": Ridge(alpha=10),
}

print("Сравнение по CV (MSE ↓ лучше):")
cv_means = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="neg_mean_squared_error"
    )
    mean_mse = -scores.mean()
    cv_means[name] = mean_mse
    print(f"  {name:20s}  CV mean MSE = {mean_mse:.2f}  (std={(-scores).std():.2f})")

# TODO-3: выбрать лучшую по CV
best_name = min(cv_means, key=cv_means.get)
print(f"\nПобедитель по CV: {best_name}")

# TODO-4: fit на 100% Train + MSE на Test
best_model = models[best_name]
best_model.fit(X_train, y_train)
test_mse = mean_squared_error(y_test, best_model.predict(X_test))
print(f"Test MSE итоговой модели: {test_mse:.2f}")
print("\n(Если меняете alpha у Ridge — делайте это по CV, а не подкручивая Test.)")


Сравнение по CV (MSE ↓ лучше):
  LinearRegression      CV mean MSE = 435.04  (std=131.58)
  Ridge(alpha=10)       CV mean MSE = 555.69  (std=59.06)

Победитель по CV: LinearRegression
Test MSE итоговой модели: 365.57

(Если меняете alpha у Ridge — делайте это по CV, а не подкручивая Test.)


### Эталон логики (не числа)

1. `train_test_split` → Test отложен.
2. `cross_val_score` **только** на `X_train, y_train`.
3. `best = argmin(mean CV MSE)`.
4. `best.fit(X_train, y_train)`.
5. `mean_squared_error(y_test, predict)` — **один** раз.

Числа у вас могут чуть отличаться — важен **порядок шагов**.

---

## Что делать дальше

1. Перечитайте ноутбук, запуская ячейки.
2. Вспомните темы **пропусков / нормировки**: любой `fit` статистик — кандидат на **Pipeline + CV**.
3. Когда будете подбирать гиперпараметры «автоматом», посмотрите `GridSearchCV` / `RandomizedSearchCV` — внутри они как раз крутят CV.
4. Для классификации замените метрику и попробуйте `StratifiedKFold`.

### Главная мысль занятия

> **Validation / CV** нужны, чтобы выбирать и крутить модель **честно**.  
> **Test** нужен, чтобы **один раз** узнать правду.  
> Перепутаете роли — получите красивую оценку и неприятный сюрприз в бою.

Удачи!
